### Block 0: Install required packages



In [1]:
!pip install -q openai datasets scikit-learn tqdm pandas


### Block 1: Imports and OpenAI client setup


In [2]:
import os
import getpass

from openai import OpenAI
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# Read your OpenAI API key (project key starting with "sk-...")
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key here: ")

# Create a single OpenAI client instance
client = OpenAI()


Paste your OpenAI API key here: ··········


### Block 2: Load FPB dataset (ChanceFocus/en-fpb, test split)

In [3]:
dataset = load_dataset("ChanceFocus/en-fpb", split="test")

print(dataset)
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/742 [00:00<?, ?B/s]

data/train-00000-of-00001-ab9a3b4799b095(…):   0%|          | 0.00/608k [00:00<?, ?B/s]

data/test-00000-of-00001-8bd1e21c671fb67(…):   0%|          | 0.00/188k [00:00<?, ?B/s]

data/valid-00000-of-00001-303e4ba2afe838(…):   0%|          | 0.00/154k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/970 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/776 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 970
})
Number of examples: 970
Example 0: {'id': 'fpb3876', 'query': 'Analyze the sentiment of this statement extracted from a financial news article. Provide your answer as either negative, positive, or neutral.\nText: The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .\nAnswer:', 'answer': 'positive', 'text': 'The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .', 'choices': ['positive', 'neutral', 'negative'], 'gold': 0}


### Block 3: Define label set and normalization helper


In [4]:
LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: str) -> str:
    """Map raw model output to one of: 'negative', 'neutral', 'positive'."""
    text = (raw or "").strip().lower()
    for label in LABELS:
        if label in text:
            return label
    return "neutral"


### Block 4: GPT-4o sentiment classifier (Chat Completions API)

In [5]:
SYSTEM_PROMPT_GPT4O = (
    "You are a financial sentiment classifier. "
    "Given a single sentence from a financial news article, classify its overall "
    "sentiment from the perspective of an investor as exactly one of: "
    "positive, neutral, or negative. "
    "Respond with only one word: positive, neutral, or negative."
)

def classify_with_gpt4o(sentence: str) -> str:
    """Call gpt-4o via Chat Completions to classify one sentence."""
    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_GPT4O},
            {"role": "user", "content": sentence},
        ],
        max_tokens=16,  # enough for a single-word answer
        temperature=0,
    )
    raw = completion.choices[0].message.content
    return normalize_prediction(raw)


### Block 5: Run evaluation loop over the FPB test set with gpt-4o

In [6]:
y_true = []
y_pred = []

for example in tqdm(dataset, total=len(dataset)):
    sentence = example["text"]
    true_label = example["answer"].lower().strip()
    pred_label = classify_with_gpt4o(sentence)

    y_true.append(true_label)
    y_pred.append(pred_label)


100%|██████████| 970/970 [12:03<00:00,  1.34it/s]


### Block 6: Compute metrics and inspect predictions

In [7]:
# Overall metrics
accuracy = accuracy_score(y_true, y_pred)
print(f"Overall accuracy: {accuracy:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS))

# Build a DataFrame with text / gold / pred
df = pd.DataFrame({
    "text": [ex["text"] for ex in dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

# Error analysis
errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))


Overall accuracy: 0.8351

Classification report:
              precision    recall  f1-score   support

    negative       0.81      0.96      0.88       116
     neutral       0.87      0.86      0.86       577
    positive       0.78      0.73      0.75       277

    accuracy                           0.84       970
   macro avg       0.82      0.85      0.83       970
weighted avg       0.83      0.84      0.83       970


First 10 predictions:
                                                                                                                                                                                                                                                                                 text     gold     pred
                                                                                           The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation function